# Part 6 — Build Your First RAG

*Five parts of theory, now one running program: a chat-with-your-documents app, built by hand.*

📖 Read the essay: https://www.mefby.com/essays/build-your-first-rag  
🐍 Companion script: `rag_app.py`

**What you'll build**

- A tiny, eyeball-able **corpus** of store-policy facts (Step 0).
- A `chunk` step that attaches **metadata** to each document (Step 1, from Part 5).
- An `embed` function backed by a local model (Step 2, from Part 2).
- An in-memory **vector store**: a list of chunks beside a NumPy matrix (Step 3, from Part 4).
- A `retrieve` function: cosine similarity + **top-k** in one `argsort` (Step 4, from Part 3).
- A grounded **prompt template** that wraps the chunks around the question (Step 5, from Part 1).
- A swappable `generate(prompt)` function isolating the LLM (Step 6).
- An `ask()` that ties the whole loop together (Step 7).
- A peek at swapping in a real vector database (Chroma).

> This notebook runs **fully offline** with **numpy only**. Where the real build calls a
> downloaded embedding model or a hosted LLM, we use it automatically *if available* and
> otherwise fall back to a transparent, deterministic, dependency-free stand-in. Every
> fallback prints a clear label so you always know which path ran.

Previous: **Part 5 — Documents & Chunking**. Next: **Part 7 — Retrieval Deep Dive**.

## Setup

We need almost nothing. `numpy` does the vector math, and the standard-library
`hashlib` / `re` power the offline embedding stand-in (the exact pattern from
Part 2). The real `sentence-transformers` and LLM SDKs are *optional*: if they're
installed we use them, and if not the notebook still runs end to end.

In [ ]:
import hashlib
import os
import re

import numpy as np

print('numpy', np.__version__)
print('Setup ready. Real models will be used automatically if available, else a transparent fallback.')

## Step 0: a tiny corpus

You cannot tell whether retrieval works unless you can check it by hand, so we
start with a knowledge base small enough to hold in your head: five short facts
about one online store's policies. This is our **corpus** (the collection of source
documents the system can draw on). When we later ask *"What is our refund window?"*
we already know the answer lives in the first document, which lets us trust the
machinery as we build it.

In [ ]:
CORPUS = [
    "Refunds are accepted within 30 days of purchase, provided the item is unused and in its original packaging.",
    "To start a return, email support@example.com with your order number. Refunds are processed within five business days of us receiving the item.",
    "Standard shipping takes 3 to 5 business days. Express shipping arrives the next business day.",
    "Shipping fees are non-refundable, and items marked final sale cannot be returned or exchanged.",
    "All electronics include a one-year limited warranty covering manufacturing defects.",
]

print(len(CORPUS), 'documents in the corpus')

## Step 1: load and chunk

In a real system you'd load PDFs or HTML and split them with a chunker from Part 5.
Our documents are each a single short sentence, so each document is already one
**chunk**. We still pass them through a `chunk` step, because that is where chunking
and **metadata** belong, and we keep a `source` field on every chunk so we can cite
it later.

In [ ]:
def chunk(corpus):
    # Each doc is already chunk-sized; we attach metadata as we go (Part 5).
    return [{"text": doc, "source": f"doc_{i}"} for i, doc in enumerate(corpus)]

chunks = chunk(CORPUS)
print(len(chunks), 'chunks')
print(chunks[0])

## Step 2: embed

Now we turn each chunk's text into a vector with a local model. This is the
embedding step from Part 2: similar meanings become nearby points. The model we'd
ship is `all-MiniLM-L6-v2` (small, fast, 384 dimensions). The headline code is
exactly what the companion script runs:

```python
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")  # small, fast, 384 dimensions

def embed(texts):
    # normalize_embeddings=True makes every vector length 1, so a dot
    # product IS cosine similarity later (the trick from Part 3).
    return model.encode(texts, normalize_embeddings=True)
```

Loading that model needs a ~90 MB download. So that this notebook runs with no
network and no cached weights, we wrap the load in `try/except` and drop to a
transparent stand-in if it fails. The real path stays the intended code; the
stand-in just keeps the cell runnable.

### The guarded embedder (real model, or transparent fallback)

This is the canonical pattern from Part 2's `embeddings.py`. `load_real_model()`
tries to load the genuine `all-MiniLM-L6-v2`; if anything goes wrong (missing
package, no network, no cached weights) it prints a clear label and returns
`None`, so we can swap in the offline fallback below.

In [ ]:
def load_real_model():
    """Return a real SentenceTransformer encoder, or None if it can't load offline."""
    try:
        from sentence_transformers import SentenceTransformer

        model = SentenceTransformer("all-MiniLM-L6-v2")  # 384 dimensions

        def encode(texts):
            # normalize_embeddings=True -> unit vectors, so a dot product
            # equals cosine similarity (the trick from Part 3).
            return np.asarray(model.encode(texts, normalize_embeddings=True))

        return encode
    except Exception as exc:  # missing package, no network, no cached weights...
        print(f"[real model unavailable: {type(exc).__name__}] -> using offline fallback")
        return None

The fallback is **not** a real embedding model. A real one is a neural network that
learned, from billions of sentences, to place similar meanings near each other.
We can't reproduce that without the model. What we *can* do, with zero dependencies,
is mimic the **interface**: any text in, a fixed-length unit vector out, deterministic
and comparable in one shared space. We hash each token into the vector's dimensions
and average. Texts that share words land closer, so the *shape* and *geometry* of the
pipeline are runnable and inspectable, while being honest that a hash has no idea that
"refund" and "reimbursement" are cousins.

In [ ]:
FALLBACK_DIM = 384  # mirror all-MiniLM-L6-v2's 384 dimensions on purpose.

def tokenize(text):
    return re.findall(r"[a-z0-9]+", text.lower())

def unit(vec):
    norm = np.linalg.norm(vec)
    return vec if norm == 0 else vec / norm

def _hashed_token_vector(token, dim):
    # Deterministic hash -> a reproducible pseudo-random direction for this token.
    h = hashlib.sha256(token.encode("utf-8")).digest()
    raw = np.frombuffer((h * (dim // len(h) + 1))[:dim], dtype=np.uint8)
    return (raw.astype(np.float64) / 255.0) * 2.0 - 1.0

def _fallback_encode_one(text, dim=FALLBACK_DIM):
    tokens = tokenize(text)
    if not tokens:
        return np.zeros(dim)
    vec = np.zeros(dim)
    for token in tokens:
        vec += _hashed_token_vector(token, dim)
    vec /= len(tokens)  # average so length doesn't depend on token count
    return unit(vec)    # unit length -> dot product == cosine (Part 3)

def fallback_encode(texts):
    return np.vstack([_fallback_encode_one(t) for t in texts])

Now we choose an encoder once: the real model if it loaded, otherwise the fallback.
Either way, `embed(texts)` is the single function the rest of the app calls, so
nothing downstream cares which path ran. (This is the same indirection we'll use for
the LLM in Step 6.)

In [ ]:
_encode = load_real_model()
USING_REAL_EMBEDDER = _encode is not None
if not USING_REAL_EMBEDDER:
    _encode = fallback_encode

def embed(texts):
    return _encode(texts)

label = 'REAL all-MiniLM-L6-v2' if USING_REAL_EMBEDDER else 'offline hashing fallback'
print(f'Embedder in use: {label}')

## Step 3: store

A "vector store" sounds heavy. At its heart it is two things kept side by side: the
**chunks** (text plus metadata) and the **matrix** of their vectors. Ours lives
entirely in program memory, an **in-memory store**, the simplest kind. We embed every
chunk now (the one-time ingestion work) and keep the text right next to the vectors:
the vector is only used to *find* a chunk; the text is what we'll actually feed the
model. That is the whole lesson of Part 4 -- keep them together.

In [ ]:
vectors = embed([c["text"] for c in chunks])  # shape: (n_chunks, 384)
print('vectors shape:', np.asarray(vectors).shape)
print('Five chunks became a (5, 384) matrix: one 384-dim vector per chunk.')

## Step 4: retrieve

This is the heart, and it is pure Part 3. Embed the query *with the same model*, score
it against every chunk vector with cosine similarity, and keep the **top-k**. Because
every vector is unit length, the cosine similarity is just a dot product, so
`vectors @ q` scores all chunks at once and `argsort` keeps the `k` highest.

The single most common beginner bug hides in line one: you must embed the query with
the *same* model you embedded the chunks with, or the vectors live in different spaces
and the scores are meaningless. Calling our shared `embed` guarantees that here.

In [ ]:
def retrieve(query, k=3):
    q = embed([query])[0]            # same model as the chunks. This matters.
    scores = vectors @ q             # dot product = cosine, because all are unit length
    top = np.argsort(-scores)[:k]    # indices of the k highest scores
    return [(chunks[i]["text"], float(scores[i])) for i in top]

for text, score in retrieve("What is our refund window?"):
    print(f"{score:.2f}  {text}")

With the **real** model the refund document wins by a mile (around `0.61`), exactly as
we predicted: a good embedding places "refund window?" next to "refunds within 30 days"
even though they share few words. With the **offline fallback** the ranking leans on
literal word overlap (it's a hash), so don't read deep meaning into the exact decimals;
it's the *shape* of the pipeline that's runnable here. Either way, it's the ranking,
not the precise numbers, that matters.

## Step 5: augment

Retrieval found the right chunks. Now we build the prompt that wraps them around the
question. We use a **prompt template**: a fixed string with slots for the retrieved
context and the user's question. Filling that context slot is **context injection**,
and we add an explicit **grounding** instruction telling the model to answer only from
the context and to admit when it cannot. That grounding line is the antidote to the
hallucination problem from Part 1 -- the single highest-leverage line in the prompt.

In [ ]:
PROMPT_TEMPLATE = """Answer the question using ONLY the context below.
If the answer is not in the context, say \"I don't know based on the provided documents.\"

Context:
{context}

Question: {question}
Answer:"""

def build_prompt(query, retrieved):
    context = "\n".join(f"- {text}" for text, _score in retrieved)
    return PROMPT_TEMPLATE.format(context=context, question=query)

print(build_prompt("What is our refund window?", retrieve("What is our refund window?")))

## Step 6: generate

The last piece is the model call, isolated behind one `generate(prompt)` function so
the provider is a single swap. One honest caveat: **LLM SDK syntax, model names, and
versions move fast.** Treat the provider lines as a snapshot and check current docs.
Keeping it all in one tiny function means a change touches one place.

Here is the intended hosted (OpenAI) default, exactly as the companion script ships it:

```python
def generate(prompt):
    from openai import OpenAI
    client = OpenAI()                  # reads OPENAI_API_KEY from your environment
    resp = client.chat.completions.create(
        model="gpt-4o-mini",           # a small, cheap chat model; check current names
        messages=[{"role": "user", "content": prompt}],
        temperature=0,                 # we want grounded, not creative
    )
    return resp.choices[0].message.content
```

Prefer to run fully local and free? Swap the body for **Ollama** and change nothing else:

```python
def generate(prompt):
    import requests
    r = requests.post(
        "http://localhost:11434/api/generate",
        json={"model": "llama3.1", "prompt": prompt, "stream": False},
    )
    return r.json()["response"]
```

Or use **Anthropic / Claude** (note: the Messages API requires `max_tokens`, and
`temperature` is omitted on Opus 4.8):

```python
def generate(prompt):
    from anthropic import Anthropic
    client = Anthropic()                # reads ANTHROPIC_API_KEY from your environment
    resp = client.messages.create(
        model="claude-opus-4-8",        # check current model names
        max_tokens=1024,                # required by the Messages API
        messages=[{"role": "user", "content": prompt}],
    )                                   # (no temperature: removed on Opus 4.8)
    return resp.content[0].text
```

### The offline `generate`

Every path above needs a key or a running local server, so none is offline-safe in a
notebook. To keep the end-to-end `ask()` runnable with **no key and no network**, we
define `generate` as a transparent, deterministic **grounded-extractive stand-in**: it
reads the same prompt and answers *only* from the injected context, pulling out the
most on-topic line (or honestly refusing when nothing matches). It is not a language
model -- it can't paraphrase -- but it obeys the exact grounding contract our prompt
asks for, so you can watch the whole loop behave correctly. It prints a clear
`[offline generate]` label so you always know the stand-in ran. In a real deployment
you'd delete this cell and use one of the provider functions above.

In [ ]:
def generate(prompt):
    # OFFLINE STAND-IN for the real LLM call shown above. It honours the SAME
    # grounding contract as the prompt: answer ONLY from the injected context,
    # otherwise say we don't know. Deterministic, no key, no network.
    print('[offline generate] grounded-extractive stand-in (no LLM call was made)')

    # Recover the context lines and the question straight out of the prompt text.
    context_block = prompt.split('Context:', 1)[-1].split('Question:', 1)[0]
    context_lines = [ln[2:].strip() for ln in context_block.splitlines()
                     if ln.strip().startswith('- ')]
    question = prompt.split('Question:', 1)[-1].split('Answer:', 1)[0].strip()

    if not context_lines:
        return "I don't know based on the provided documents."

    # Score each context line by word overlap with the question (a transparent,
    # dependency-free proxy for 'most relevant sentence').
    q_words = set(tokenize(question)) - {'is', 'our', 'the', 'a', 'an', 'of', 'do', 'you', 'what', 'how', 'when'}
    def overlap(line):
        return len(q_words & set(tokenize(line)))
    best = max(context_lines, key=overlap)

    # If even the best line shares no meaningful words, refuse -- exactly what the
    # grounding instruction asks for, and the whole point of RAG done honestly.
    if overlap(best) == 0:
        return "I don't know based on the provided documents."
    return best

## Step 7: wrap it into an app

Three lines tie the loop together: retrieve, augment, generate. That `ask()` *is* the
whole RAG pipeline. In the companion script a tiny **REPL** (a read-eval-print loop)
wraps `ask()` so it feels like a real app; in a notebook we just call `ask()` directly.

In [ ]:
def ask(question, k=3):
    retrieved = retrieve(question, k=k)          # Part 3 + 4: find relevant chunks
    prompt = build_prompt(question, retrieved)   # Part 5 + 1: ground the prompt
    return generate(prompt)                      # the LLM (here: offline stand-in) answers

# The companion script's REPL, shown for reference (don't run it here -- input() blocks):
#
# if __name__ == "__main__":
#     print("Ask about the store policy (Ctrl-C to quit).\n")
#     while True:
#         try:
#             q = input("> ").strip()
#             if q:
#                 print(ask(q), "\n")
#         except (EOFError, KeyboardInterrupt):
#             print("\nBye.")
#             break
print('ask() defined.')

## The headline demo

Now run the companion script's demo through the offline path. We ask one question whose
answer *is* in the corpus and one whose answer is *not*. The second is the whole point:
the corpus says nothing about gift wrapping, so the grounded pipeline refuses instead of
inventing a policy. That honest "I don't know" is RAG working as designed.

In [ ]:
for q in ["What is our refund window?", "Do you offer gift wrapping?"]:
    print(f"> {q}")
    print(ask(q))
    print()

## The upgrade: swap in a real vector database

The hand-rolled store taught us exactly what a vector store does. In production you'd
let a real one do the embedding, storage, and indexing (the work of Part 4). Here is the
same app with **Chroma**, a local embedded database, dropped in. Watch how `add` and
`query` collapse our manual embed-store-cosine code into two calls. (Shown as reference
code -- it needs `pip install chromadb` and downloads the model, so we don't run it in
this offline notebook.)

```python
import chromadb
from chromadb.utils import embedding_functions

client = chromadb.Client()   # in-memory; use PersistentClient(path="...") to keep it
ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"        # the SAME model as before
)
collection = client.create_collection(
    "policy",
    embedding_function=ef,
    metadata={"hnsw:space": "cosine"},   # match the cosine metric we used by hand
)

# add() embeds and stores in one call; ids and metadata travel with the text
collection.add(
    documents=[c["text"] for c in chunks],
    ids=[c["source"] for c in chunks],
)

def retrieve(query, k=3):
    res = collection.query(query_texts=[query], n_results=k)
    # Chroma returns distances, where smaller means closer (Part 4's metrics)
    return list(zip(res["documents"][0], res["distances"][0]))
```

That is the only change. `ask`, `build_prompt`, and `generate` are untouched, because
the interface (give me a query, get back relevant chunks) is the same. One trap to mind:
Chroma reports a **distance** (smaller is closer), not a **similarity** (larger is
closer), so if you ever sort or threshold on the number, watch the direction -- exactly
the similarity-versus-distance trap from Part 3.

## What you learned

- A RAG app is one short loop: **retrieve** relevant chunks, **augment** the prompt with
  them, **generate** an answer. We built it by hand so every line maps to a concept from
  Parts 1 through 5.
- Use a **local embedding model** and **embed the query with the same model** as the
  chunks. Our shared `embed` enforces that, preventing the most common silent failure.
- The simplest **vector store** is a list of chunks plus a NumPy matrix; cosine **top-k**
  retrieval is one `argsort` over a dot product (Part 3). Keep the chunk text beside its
  vector.
- A **prompt template** with a **grounding** instruction is the highest-leverage line for
  curbing hallucination (Part 1). Our offline `generate` honours that same contract.
- Isolate the LLM behind one `generate(prompt)` function so switching providers (OpenAI,
  Ollama, Anthropic) -- or a test stand-in -- is a one-place change.
- Graduating to a real vector database like **Chroma** swaps storage and indexing without
  touching the rest of the app, because the retrieve interface stays the same (Part 4).

### A note on what just ran

To stay runnable with no network and no API key, this notebook used the **real**
`all-MiniLM-L6-v2` for embeddings *if it was installed and cached*, otherwise a
deterministic **hashing fallback**; and it always used an **offline grounded-extractive**
`generate` stand-in instead of a live LLM. In production you'd keep the headline embed
and swap `generate` for one of the provider functions in Step 6.

### Next

Our app uses plain dense retrieval with a fixed `k` and a fixed prompt. That works
surprisingly well, and it's also where the interesting engineering begins. What if the
best chunk uses different words than the question? What if you want to combine keyword
search with semantic search? How should `k` adapt to the query? That's the retrieval deep
dive: **Part 7 — Retrieval Deep Dive**.